# C03 回测闭环与 RQAlpha 模板（在线版）

使用在线行情做手写回测，并保留 RQAlpha 模板。


In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "601318.XSHG", "600519.XSHG"]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


In [ ]:
close = rqdatac.get_price(
    UNIVERSE,
    start_date=START_DATE,
    end_date=END_DATE,
    fields="close",
    adjust_type="post",
    expect_df=False,
)
close = close.droplevel(0, axis=1) if isinstance(close.columns, pd.MultiIndex) else close
returns = close.pct_change().fillna(0)
close.head()


## 手写回测：月末调仓 + 动量 top2 等权


In [ ]:
def rebalance_dates(index):
    return pd.Series(1, index=index).resample("ME").last().index


def target_weights(price_df, as_of_date, top_n=2):
    hist = price_df.loc[:as_of_date].tail(21)
    mom = hist.iloc[-1] / hist.iloc[0] - 1
    pick = mom.sort_values(ascending=False).head(top_n).index
    w = pd.Series(0.0, index=price_df.columns)
    w.loc[pick] = 1.0 / top_n
    return w


def simulate(price_df, fee=0.001, slip=0.0005):
    rets = price_df.pct_change().fillna(0)
    rbd = rebalance_dates(price_df.index)

    weights = pd.DataFrame(0.0, index=price_df.index, columns=price_df.columns)
    turnover = pd.Series(0.0, index=price_df.index)
    w = pd.Series(0.0, index=price_df.columns)

    for d in price_df.index:
        if d in rbd:
            tw = target_weights(price_df, d)
            turnover.loc[d] = (tw - w).abs().sum()
            w = tw
        weights.loc[d] = w

    gross = (weights.shift(1).fillna(0) * rets).sum(axis=1)
    net = gross - turnover * (fee + slip)
    wealth = (1 + net).cumprod()

    return pd.DataFrame({"gross": gross, "net": net, "turnover": turnover, "wealth": wealth})

bt = simulate(close)
bt.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bt["wealth"].plot(ax=ax, title="C03 在线版策略净值")
ax.set_ylabel("NAV")
plt.show()

ann_ret = bt["net"].mean() * 252
ann_vol = bt["net"].std() * np.sqrt(252)
summary = pd.DataFrame({
    "annual_return": [ann_ret],
    "annual_vol": [ann_vol],
    "sharpe": [ann_ret / (ann_vol + 1e-12)],
    "max_drawdown": [(bt["wealth"] / bt["wealth"].cummax() - 1).min()],
    "avg_turnover": [bt["turnover"].mean()],
}).round(4)
summary


## RQAlpha 模板（在线）


In [ ]:
# from rqalpha import run_func
#
# def init(context):
#     context.pool = ["000001.XSHE", "000002.XSHE"]
#
# def handle_bar(context, bar_dict):
#     # 在这里写在线信号 + 下单逻辑
#     pass
#
# config = {
#     "base": {
#         "start_date": "2022-01-01",
#         "end_date": "2024-12-31",
#         "frequency": "1d",
#         "accounts": {"stock": 1_000_000},
#     },
#     "mod": {"sys_analyser": {"enabled": True}},
# }
#
# result = run_func(init=init, handle_bar=handle_bar, config=config)
